# 02 – Utforsk MNIST

**Tid:** 60 minutter. **Mål:** forstå hvordan tall blir til bilder,
undersøke bilder og etiketter og visualisere klassefordelingen.

Vi starter med en liten matrise vi kjenner, og går så videre til MNIST:
håndskrevne sifre med 28 × 28 piksler. Vi bruker et fast tilfeldig utvalg
på 6 000 treningsbilder og 1 000 testbilder. Bildene du velger å vise,
trekkes tilfeldig og kan være forskjellige fra sidemannens.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn


## Et gråtonebilde er en matrise med tall

Et digitalt bilde består av små ruter kalt **piksler**. I et gråtonebilde
beskriver ett tall lysstyrken til hver piksel. Tallene er ordnet i rader
og kolonner: en **matrise**, som vi kan lagre i en todimensjonal tensor.
Det er disse tallene modellen får, ikke en egen forståelse av et bilde.

Her lager vi et bilde med fire rader og fire kolonner. Med skalaen 0–1
betyr 0 svart, 1 hvitt og verdier imellom grått. `image[row, column]`
velger én piksel. Rad 0 ligger øverst, kolonne 0 lengst til venstre.
Et fargebilde trenger flere verdier per piksel, vanligvis rødt, grønt og
blått; MNIST har bare gråtoner.

Kjør eksemplet. Sammenlign tallmatrisen med figuren: `imshow` oversetter
hvert tall til en lysstyrke. Tallene i rutene viser koblingen direkte.


In [ ]:
example_image = torch.tensor([
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 0.4, 0.8, 0.0],
    [0.0, 0.8, 1.0, 0.0],
    [0.0, 0.0, 0.0, 0.0],
])
print("Bildet som tallmatrise:")
print(example_image)
print("Piksel i rad 1, kolonne 2:", example_image[1, 2].item())

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(example_image, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
for row in range(4):
    for column in range(4):
        value = example_image[row, column].item()
        ax.text(column, row, f"{value:.1f}", ha="center", va="center",
                  color="black" if value > 0.5 else "white")
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set(xlabel="Kolonne", ylabel="Rad", title="De samme tallene vist som et bilde")
plt.show()


## Oppgave 1: Endre én piksel

Før du kjører: Hva tror du skjer hvis pikselen i rad 0, kolonne 0 endres
fra 0 til 1? Lag `changed_image = example_image.clone()`; `clone()` lager
en uavhengig kopi. Endre denne pikselen til 1 og vis det nye bildet med
`imshow`, gråtoner og samme faste skala (`vmin=0, vmax=1`).
Skriv også ut den nye matrisen.

<details>
<summary>Hint</summary>

`changed_image[0, 0] = 1.0` endrer øverste venstre piksel.
Bruk figurkoden fra eksemplet; tallene i rutene er valgfrie i din figur.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør cellen når du har endret bildet.


In [ ]:
assert changed_image.shape == example_image.shape
assert changed_image[0, 0].item() == 1.0
assert example_image[0, 0].item() == 0.0
assert (changed_image != example_image).sum().item() == 1


### Sjekk forståelsen

Skriv hva du forventet før kjøring, og hva du observerte.
Hvor i bildet skjedde endringen, og hvorfor akkurat der?
Har matrisen og bildet forskjellig informasjon, eller er de to visninger
av de samme verdiene? Forklar også hvorfor lysstyrke 0.8 ikke betyr
at modellen er 80 % sikker på et siffer.

Skriv svaret ditt i Markdown-cellen under.


**Mitt svar:**

Skriv din forklaring her.


## Ferdig kode: last ned og hent datasettet

Nå går vi fra den lille matrisen til håndskrevne sifre. Første gang
lastes MNIST ned til prosjektets `data/`-mappe. Senere brukes de samme
lokale filene. Hele MNIST lastes ned, men vi bruker bare delutvalget.

Et **datasett** lar oss hente ett par `(image, label)` med en indeks,
omtrent som fra en liste. `image` inneholder pikselverdiene, og `label`
forteller hvilket siffer bildet forestiller. Vi undersøker ett bilde
om gangen før vi prøver å lære fra dem.


In [ ]:
from pytorch_101.utils import download_mnist

train_data, test_data = download_mnist()
print(f"Treningsbilder: {len(train_data)}, testbilder: {len(test_data)}")


## Samme idé, flere piksler

Rå MNIST-bilder er matriser med 28 rader og 28 kolonner. Hvert tall er
en pikselverdi fra 0 til 255. For å se et siffer velger vi gråtoner for
disse verdiene; akkurat som i 4 × 4-eksemplet, bare med flere ruter.

Datahjelperen bruker `ToTensor`, som skalerer tallene til 0–1 og legger
til en **kanalakse**. Derfor har bildet formen `[1, 28, 28]`:
én gråtonekanal, 28 rader og 28 kolonner. Etiketten lagres separat.
`image[0, row, column]` henter én piksel fra denne ene kanalen.

Cellen under henter råbildet og den skalerte versjonen av samme bilde.
Legg merke til at en liten utskjæring av bildet også bare er tall.


In [ ]:
image, label = train_data[0]
original_index = train_data.indices[0]
raw_image = train_data.dataset.data[original_index]
print("Råbilde:", raw_image.shape, raw_image.dtype, raw_image.min().item(), raw_image.max().item())
print("Modellens bilde:", image.shape, image.dtype, image.min().item(), image.max().item())
print("Etikett:", label)

print("Et utsnitt av pikselmatrisen:")
print(image[0, 10:16, 10:16])


## Oppgave 2: Vis et MNIST-bilde

Trekk en tilfeldig indeks i `train_data`, og hent bilde og etikett derfra.
Kall dem `my_image` og
`my_label`. Vis bildet i gråtoner med etiketten i tittelen, uten akser.
Skriv også ut form, datatype og minste og største pikselverdi.

<details>
<summary>Hint</summary>

`torch.randint(len(train_data), ()).item()` trekker en indeks.
`my_image.squeeze(0)` endrer formen fra `[1, 28, 28]` til `[28, 28]`.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert my_image.shape == (1, 28, 28)
assert 0 <= my_label <= 9
assert 0 <= my_image.min() <= my_image.max() <= 1


### Sjekk forståelsen

Hva betyr dimensjonene i formen til `my_image`? Hvorfor fjerner vi
første dimensjon ved visning? Hva betyr svart og hvitt med denne fargeskalaen?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Skriv din forklaring her.


## Eksempel: flere bilder i samme figur

`subplots` lager flere akser. `ravel()` gir oss en flat liste av aksene, slik
at vi kan bruke en vanlig løkke. Hver akse får sitt eget bilde og sin egen tittel.


In [ ]:
example_images = [example_image, 1 - example_image,
                  example_image.T, torch.zeros_like(example_image)]
fig, axes = plt.subplots(2, 2, figsize=(5, 5))
for index, ax in enumerate(axes.ravel()):
    ax.imshow(example_images[index], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Eksempel {index}")
    ax.axis("off")
fig.tight_layout()
plt.show()


## Valgfri oppgave 3: Lag et bilderutenett

Lag en figur med 3 × 3 akser. Trekk ni forskjellige treningsbilder og vis dem med riktig
etikett over hvert bilde. Bruk samme gråtoneskala i alle rutene.

<details>
<summary>Hint</summary>

Bruk `torch.randperm(len(train_data))[:9].tolist()` for ni ulike indekser.
Kombiner dem med aksene ved hjelp av `zip(axes.ravel(), image_indices)`.

</details>


In [ ]:
# Skriv din kode her.


## Eksempel: tell kategorier og tegn et stolpediagram

Klasseetiketter er kategorier. Derfor bruker vi ett stolpediagram med én stolpe
per kategori. Vi starter med en liten liste frukter og en vanlig Python-løkke.


In [ ]:
fruits = ["eple", "pære", "eple", "banan", "pære", "eple"]
categories = ["eple", "pære", "banan"]
fruit_counts = [fruits.count(category) for category in categories]
fig, ax = plt.subplots()
ax.bar(categories, fruit_counts)
ax.set(xlabel="Frukt", ylabel="Antall", title="Fordeling av frukter")
plt.show()


## Oppgave 4: Undersøk klassefordelingen

Tell hvor mange bilder det er av hvert siffer i **treningsutvalget**.
Lag listen `class_counts` med ti tall, der indeks 0 er antall nuller.
Lag et stolpediagram med siffer på x-aksen og antall bilder på y-aksen.
Vis alle sifrene 0–9 langs x-aksen. Skriv også ut antallene.

<details>
<summary>Hint</summary>

Start med `class_counts = [0] * 10`. For hvert `image, label` i datasettet:
øk `class_counts[label]` med 1. Bruk `range(10)` som stolpenes posisjoner.

</details>

Finn også `majority_class` og `baseline_accuracy`: hvilken nøyaktighet får
en modell som alltid gjetter det vanligste treningssifferet, målt på treningsutvalget?


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert len(class_counts) == 10
assert sum(class_counts) == len(train_data)
assert all(count > 0 for count in class_counts)

assert majority_class == max(range(10), key=lambda digit: class_counts[digit])
assert baseline_accuracy == max(class_counts) / len(train_data)


### Sjekk forståelsen

Hvilket siffer forekommer oftest og sjeldnest? Oppgi begge antallene fra din kjøring.
Er fordelingen omtrent jevn? Hvorfor kan en svært skjev fordeling være et problem
ved trening og når vi rapporterer samlet nøyaktighet?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.

Hvorfor er flertallsgjetting en nyttig referanse? Er høy treningsnøyaktighet nok til å vise at en modell fungerer på nye bilder?


**Mitt svar:**

Skriv din forklaring her.


**Neste steg:** Vi skal bruke pikselverdiene til å beregne skårer for de ti sifrene.


## Videre lesing

Lenkene er på engelsk og er valgfrie oppslag eller fordypning. Du trenger ikke lese alt for å løse oppgavene.

- **Start her:** [PyTorch – Datasets & DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html). Les først om å hente og vise bilder fra et datasett. Eksemplet bruker FashionMNIST, som inneholder klær; vi bruker MNIST-sifre. Delen om `DataLoader` passer som forberedelse til neste notatbok.
- **Oppslagsverk:** [Torchvision – MNIST](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html). Offisiell dokumentasjon for datasettet bak `download_mnist()`. Se særlig hva `train`, `transform` og `download` betyr.
